In [ ]:
import sys
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

sys.path.insert(0, str(Path.cwd().parent))
%load_ext autoreload
%autoreload 2

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from typing import Dict, Any
from backtesting import Backtest

from strat.s_atr_breakout import (
    load_data,
    build_features,
    convert_to_ohlcv,
    AtrBreakoutStrategy,
)
from optim.optim_func_params import grid_search, OptimMetric

pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

%matplotlib inline

In [ ]:
SYMBOL = "EURUSD"
START_DATE = "20150101"
END_DATE = "20161231"

print(f"Loading {SYMBOL} data from {START_DATE} to {END_DATE}...")
df_raw = load_data(p_symbol=SYMBOL, p_start=START_DATE, p_end=END_DATE)
print(f"Loaded {len(df_raw):,} bars")
print(f"Date range: {df_raw.index[0]} to {df_raw.index[-1]}")

In [ ]:
def run_atr_backtest(df: pd.DataFrame, params: Dict[str, Any]) -> Dict[str, float]:
    """
    Run ATR breakout backtest with given parameters.
    
    Args:
        df: Raw OHLCV DataFrame
        params: Dict with keys:
            - atr_mult: ATR multiplier for breakout levels
            - stop_mult: Stop loss in ATR units
            - atr_len: ATR calculation period
            - gap_threshold: Minimum gap % for directional bias
            - use_gap_bias: Whether to use gap bias (0 or 1)
            - use_vol_filter: Whether to use vol filter (0 or 1)
    
    Returns:
        Dict with metrics: sharpe_ratio, total_return, profit_factor, etc.
    """
    df = df.copy()
    
    atr_mult = params['atr_mult']
    stop_mult = params['stop_mult']
    atr_len = params['atr_len']
    gap_threshold = params['gap_threshold']
    use_gap_bias = bool(params['use_gap_bias'])
    use_vol_filter = bool(params['use_vol_filter'])
    
    df = build_features(
        df,
        p_atr_len=atr_len,
        p_atr_mult=atr_mult,
        p_use_gap_bias=use_gap_bias,
        p_use_vol_filter=use_vol_filter,
        p_gap_threshold=gap_threshold,
        p_direction="both",
    )
    
    ohlcv_df = convert_to_ohlcv(df)
    
    if len(ohlcv_df) == 0:
        return {
            'sharpe_ratio': np.nan,
            'calmar_ratio': np.nan,
            'total_return': np.nan,
            'profit_factor': np.nan,
            'max_drawdown': np.nan,
            'win_rate': np.nan,
            'n_trades': 0,
        }
    
    bt = Backtest(
        ohlcv_df,
        AtrBreakoutStrategy,
        cash=100_000,
        commission=0.00002,
        trade_on_close=True,
        hedging=False,
        exclusive_orders=False,
        finalize_trades=True,
    )
    
    stats = bt.run(
        atr_mult=atr_mult,
        stop_mult=stop_mult,
        exit_eod=True,
        direction="both",
    )
    
    return {
        'sharpe_ratio': float(stats['Sharpe Ratio']) if np.isfinite(stats['Sharpe Ratio']) else -999.0,
        'calmar_ratio': float(stats['Calmar Ratio']) if np.isfinite(stats['Calmar Ratio']) else -999.0,
        'total_return': float(stats['Return [%]']),
        'profit_factor': float(stats['Profit Factor']) if np.isfinite(stats['Profit Factor']) else 0.0,
        'max_drawdown': float(stats['Max. Drawdown [%]']),
        'win_rate': float(stats['Win Rate [%]']) if np.isfinite(stats['Win Rate [%]']) else 0.0,
        'n_trades': int(stats['# Trades']),
    }

print("Backtest wrapper function defined.")

In [ ]:
test_params = {
    'atr_mult': 0.5,
    'stop_mult': 0.5,
    'atr_len': 14,
    'gap_threshold': 0.002,
    'use_gap_bias': 0,
    'use_vol_filter': 0,
}

print("Testing single backtest...")
test_result = run_atr_backtest(df_raw, test_params)
print(f"Result: {test_result}")

In [ ]:
PARAM_RANGES = {
    'atr_mult': [0.3, 0.5, 0.7, 1.0],
    'stop_mult': [0.3, 0.5, 0.7, 1.0],
    'atr_len': [10, 14, 20],
    'gap_threshold': [0.001, 0.002, 0.003],
    'use_gap_bias': [0, 1],
    'use_vol_filter': [0, 1],
}

from itertools import product

all_combinations = list(product(*PARAM_RANGES.values()))
total_combinations = len(all_combinations)
print(f"Total possible combinations: {total_combinations}")

MAX_COMBINATIONS = 10
limited_combinations = all_combinations[:MAX_COMBINATIONS]
print(f"Using first {MAX_COMBINATIONS} combinations")

param_ranges_limited = {
    'atr_mult': [],
    'stop_mult': [],
    'atr_len': [],
    'gap_threshold': [],
    'use_gap_bias': [],
    'use_vol_filter': [],
}

for combo in limited_combinations:
    for i, key in enumerate(param_ranges_limited.keys()):
        if combo[i] not in param_ranges_limited[key]:
            param_ranges_limited[key].append(combo[i])

print(f"\nLimited parameter ranges:")
for key, values in param_ranges_limited.items():
    print(f"  {key}: {values}")

In [ ]:
print("Running grid search optimization...")
print(f"Symbol: {SYMBOL}")
print(f"Period: {START_DATE} to {END_DATE}")
print(f"Combinations: {len(limited_combinations)}")
print("-" * 60)

results_df, best_result, pivot_df = grid_search(
    func=run_atr_backtest,
    df=df_raw,
    param_ranges=param_ranges_limited,
    optim_metric=OptimMetric.SHARPE,
    maximize=True,
    verbose=True,
)

print("\n" + "=" * 60)
print("OPTIMIZATION COMPLETE")
print("=" * 60)

In [ ]:
print("\n--- BEST PARAMETERS ---")
for key, value in best_result.params.items():
    print(f"  {key}: {value}")

print("\n--- BEST METRICS ---")
print(f"  Sharpe Ratio:  {best_result.sharpe:.4f}")
print(f"  Total Return:  {best_result.total_return:.2f}%")
print(f"  Profit Factor: {best_result.profit_factor:.4f}")
print(f"  Calmar Ratio:  {best_result.calmar:.4f}")
print(f"  Max Drawdown:  {best_result.metrics['max_drawdown']:.2f}%")
print(f"  Win Rate:      {best_result.metrics['win_rate']:.2f}%")
print(f"  # Trades:      {best_result.metrics['n_trades']}")

In [ ]:
print("\n--- ALL RESULTS (sorted by Sharpe) ---")
results_sorted = results_df.sort_values('sharpe_ratio', ascending=False)
display_cols = ['atr_mult', 'stop_mult', 'atr_len', 'gap_threshold', 'use_gap_bias', 'use_vol_filter', 
                'sharpe_ratio', 'total_return', 'profit_factor', 'win_rate', 'n_trades']
print(results_sorted[display_cols].head(10).to_string())

In [ ]:
if len(pivot_df) > 0 and not pivot_df.empty:
    plt.figure(figsize=(10, 8))
    sns.heatmap(pivot_df, annot=True, fmt='.2f', cmap='RdYlGn', center=0)
    plt.title(f'Sharpe Ratio: {list(pivot_df.columns.name)} vs {pivot_df.index.name}')
    plt.tight_layout()
    plt.show()
else:
    print("No 2D pivot table available (need exactly 2 parameters with multiple values)")
    
    if len(results_df) > 0:
        fig, axes = plt.subplots(2, 2, figsize=(12, 10))
        
        ax1 = axes[0, 0]
        results_df.plot.scatter(x='atr_mult', y='sharpe_ratio', ax=ax1)
        ax1.set_title('Sharpe vs ATR Multiplier')
        ax1.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
        
        ax2 = axes[0, 1]
        results_df.plot.scatter(x='stop_mult', y='sharpe_ratio', ax=ax2)
        ax2.set_title('Sharpe vs Stop Multiplier')
        ax2.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
        
        ax3 = axes[1, 0]
        results_df.plot.scatter(x='total_return', y='sharpe_ratio', ax=ax3)
        ax3.set_title('Sharpe vs Total Return')
        ax3.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
        ax3.axvline(x=0, color='gray', linestyle='--', alpha=0.5)
        
        ax4 = axes[1, 1]
        results_df.plot.scatter(x='n_trades', y='sharpe_ratio', ax=ax4)
        ax4.set_title('Sharpe vs Number of Trades')
        ax4.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
        
        plt.tight_layout()
        plt.show()

In [ ]:
TRAIN_START = "20150101"
TRAIN_END = "20151231"
TEST_START = "20160101"
TEST_END = "20161231"

print("--- TRAIN/TEST VALIDATION ---")
print(f"Train: {TRAIN_START} to {TRAIN_END}")
print(f"Test:  {TEST_START} to {TEST_END}")

print("\nLoading train data...")
df_train = load_data(p_symbol=SYMBOL, p_start=TRAIN_START, p_end=TRAIN_END)
print(f"Train bars: {len(df_train):,}")

print("\nLoading test data...")
df_test = load_data(p_symbol=SYMBOL, p_start=TEST_START, p_end=TEST_END)
print(f"Test bars: {len(df_test):,}")

In [ ]:
best_params = best_result.params
print("\n--- VALIDATING BEST PARAMETERS ---")
print(f"Params: {best_params}")

print("\nRunning on TRAIN data...")
train_result = run_atr_backtest(df_train, best_params)
print(f"  Sharpe: {train_result['sharpe_ratio']:.4f}")
print(f"  Return: {train_result['total_return']:.2f}%")
print(f"  # Trades: {train_result['n_trades']}")

print("\nRunning on TEST data...")
test_result = run_atr_backtest(df_test, best_params)
print(f"  Sharpe: {test_result['sharpe_ratio']:.4f}")
print(f"  Return: {test_result['total_return']:.2f}%")
print(f"  # Trades: {test_result['n_trades']}")

print("\n--- TRAIN vs TEST COMPARISON ---")
comparison = pd.DataFrame({
    'Metric': ['Sharpe', 'Return %', 'Profit Factor', 'Win Rate %', 'Max DD %', '# Trades'],
    'Train': [
        train_result['sharpe_ratio'],
        train_result['total_return'],
        train_result['profit_factor'],
        train_result['win_rate'],
        train_result['max_drawdown'],
        train_result['n_trades'],
    ],
    'Test': [
        test_result['sharpe_ratio'],
        test_result['total_return'],
        test_result['profit_factor'],
        test_result['win_rate'],
        test_result['max_drawdown'],
        test_result['n_trades'],
    ],
})
print(comparison.to_string(index=False))

In [ ]:
def test_symbol(symbol: str, start: str, end: str, params: Dict[str, Any]) -> Dict[str, float]:
    """Test strategy on a different symbol."""
    try:
        df = load_data(p_symbol=symbol, p_start=start, p_end=end)
        result = run_atr_backtest(df, params)
        return result
    except Exception as e:
        print(f"Error loading {symbol}: {e}")
        return None

OTHER_SYMBOLS = ["GBPUSD", "AUDUSD"]

print("\n--- TESTING ON OTHER SYMBOLS ---")
print(f"Using best params from {SYMBOL}: {best_params}")
print()

for sym in OTHER_SYMBOLS:
    print(f"Testing {sym}...")
    result = test_symbol(sym, START_DATE, END_DATE, best_params)
    if result:
        print(f"  Sharpe: {result['sharpe_ratio']:.4f}")
        print(f"  Return: {result['total_return']:.2f}%")
        print(f"  # Trades: {result['n_trades']}")
    print()

In [ ]:
OUTPUT_DIR = Path('../data_work/optim_results')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

timestamp = pd.Timestamp.now().strftime('%Y%m%d_%H%M%S')
output_file = OUTPUT_DIR / f'atr_breakout_optim_{SYMBOL}_{timestamp}.parquet'

results_df.to_parquet(output_file)
print(f"Results saved to: {output_file}")

best_params_file = OUTPUT_DIR / f'atr_breakout_best_params_{SYMBOL}_{timestamp}.txt'
with open(best_params_file, 'w') as f:
    f.write(f"# ATR Breakout Best Parameters\n")
    f.write(f"# Symbol: {SYMBOL}\n")
    f.write(f"# Period: {START_DATE} to {END_DATE}\n")
    f.write(f"# Optimized: {pd.Timestamp.now()}\n\n")
    for key, value in best_result.params.items():
        f.write(f"{key}={value}\n")
    f.write(f"\n# Metrics\n")
    f.write(f"sharpe_ratio={best_result.sharpe}\n")
    f.write(f"total_return={best_result.total_return}\n")
    f.write(f"profit_factor={best_result.profit_factor}\n")
print(f"Best params saved to: {best_params_file}")